# Exploratory Data Analysis (EDA) - RACE Dataset

This notebook explores the RACE dataset to understand distributions, class balance, passage characteristics, and question types before training.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")

## 1. Load Data

In [ ]:
raw_df = pd.read_csv('../data/raw/train.csv')
print(f"Raw dataset shape: {raw_df.shape}")
print(f"Columns: {list(raw_df.columns)}")
display(raw_df.head())

In [ ]:
processed_df = pd.read_csv('../data/processed/train_binary.csv')
print(f"Processed dataset shape: {processed_df.shape}")
display(processed_df.head())

## 2. Summary Statistics

In [ ]:
print("=== Raw Dataset Summary ===")
print(f"Total questions: {len(raw_df)}")
print(f"Unique articles: {raw_df['article'].nunique()}")
print(f"Missing values:\n{raw_df.isnull().sum()}")
print(f"\nAnswer distribution:\n{raw_df['answer'].value_counts()}")

## 3. Answer Distribution (A/B/C/D Balance)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Answer label distribution
ans_counts = raw_df['answer'].value_counts().sort_index()
axes[0].bar(ans_counts.index, ans_counts.values, color=['#4CAF50', '#2196F3', '#FF9800', '#F44336'])
axes[0].set_title('Answer Label Distribution (A/B/C/D)')
axes[0].set_xlabel('Answer Label')
axes[0].set_ylabel('Count')

# Binary class imbalance in processed data
label_counts = processed_df['label'].value_counts().sort_index()
axes[1].bar(['Incorrect (0)', 'Correct (1)'], label_counts.values, color=['#F44336', '#4CAF50'])
axes[1].set_title('Binary Label Distribution (Processed)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Class ratio: {label_counts[0]/label_counts[1]:.1f}:1 (Incorrect:Correct)")

## 4. Article and Question Lengths

In [ ]:
raw_df['article_len'] = raw_df['article'].dropna().apply(lambda x: len(str(x).split()))
raw_df['question_len'] = raw_df['question'].dropna().apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(raw_df['article_len'], bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Article Word Counts')
axes[0].set_xlabel('Words')
axes[0].axvline(raw_df['article_len'].median(), color='red', linestyle='--', label=f"Median: {raw_df['article_len'].median():.0f}")
axes[0].legend()

sns.histplot(raw_df['question_len'], bins=30, ax=axes[1], color='coral')
axes[1].set_title('Distribution of Question Word Counts')
axes[1].set_xlabel('Words')
axes[1].axvline(raw_df['question_len'].median(), color='red', linestyle='--', label=f"Median: {raw_df['question_len'].median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Article length — Mean: {raw_df['article_len'].mean():.0f}, Median: {raw_df['article_len'].median():.0f}, Max: {raw_df['article_len'].max()}")
print(f"Question length — Mean: {raw_df['question_len'].mean():.0f}, Median: {raw_df['question_len'].median():.0f}, Max: {raw_df['question_len'].max()}")

## 5. Option Lengths

In [ ]:
for opt in ['A', 'B', 'C', 'D']:
    raw_df[f'{opt}_len'] = raw_df[opt].dropna().apply(lambda x: len(str(x).split()))

opt_lens = pd.DataFrame({
    'Option': ['A', 'B', 'C', 'D'],
    'Mean Length': [raw_df[f'{o}_len'].mean() for o in ['A', 'B', 'C', 'D']],
    'Median Length': [raw_df[f'{o}_len'].median() for o in ['A', 'B', 'C', 'D']],
})
display(opt_lens)

## 6. Question Type Analysis

Classify questions by their starting word to understand question type distribution.

In [ ]:
def get_question_type(q):
    q = str(q).strip().lower()
    for wh in ['what', 'who', 'where', 'when', 'why', 'how', 'which']:
        if q.startswith(wh):
            return wh.capitalize()
    if '_' in q or '____' in q:
        return 'Fill-in-blank'
    return 'Other'

raw_df['q_type'] = raw_df['question'].apply(get_question_type)

plt.figure(figsize=(10, 5))
type_counts = raw_df['q_type'].value_counts()
type_counts.plot(kind='bar', color='teal')
plt.title('Question Type Distribution')
plt.xlabel('Question Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(type_counts)

## 7. Handcrafted Feature Distributions (Processed Data)

In [ ]:
if 'overlap_ratio' in processed_df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    sns.histplot(data=processed_df, x='overlap_ratio', hue='label', bins=30, ax=axes[0])
    axes[0].set_title('Overlap Ratio by Label')
    
    sns.histplot(data=processed_df, x='option_len', hue='label', bins=30, ax=axes[1])
    axes[1].set_title('Option Length by Label')
    
    sns.histplot(data=processed_df, x='option_position', hue='label', bins=30, ax=axes[2])
    axes[2].set_title('Option Position by Label')
    
    plt.tight_layout()
    plt.show()
else:
    print('Handcrafted features not found. Run preprocessing.py first.')